# Reconnaissance de $X_t$, façon TP

Notebook bac à sable pour tester des modèles de $X_t$, dans l'esprit du TP8 (reconnaissance de processus)
et du TP7 (simuler une EDS et la valider). On traite la série $X$ extraite comme un fichier de données à
reconnaître, et on essaie des EDS candidates avec les outils des TP.

In [1]:
import numpy as np
import scipy.stats as scs
import matplotlib.pyplot as plt

## 1. La série à reconnaître

On extrait $X$ de `virus4` comme au modèle 1. C'est notre jeu de données.

In [2]:
data = np.loadtxt('virus4.csv', delimiter=',')
V, P = data[:, 0], data[:, 1]
N = len(data)
num = P[1:] - P[:-1]; den = P[:-1] * (V[:-1] - 1); ok = np.abs(den) > 1e-3
dt = round(np.median(num[ok] / den[ok]), 3)
tau = N * dt
X = (V[1:] - V[:-1]) / (V[:-1] * dt) - 2/3 + 4/3 * P[:-1]
print('N=%d  dt=%.3f  tau=%.0f' % (len(X), dt, tau))
print('X : moyenne=%.3f  ecart-type=%.3f  plage=[%.2f, %.2f]' % (X.mean(), X.std(), X.min(), X.max()))

N=1999  dt=0.015  tau=30
X : moyenne=-0.395  ecart-type=0.228  plage=[-0.71, -0.00]


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(np.arange(len(X)) * dt, X)
plt.xlabel('temps t'); plt.ylabel('X_t')
plt.show()

## 2. Les outils des TP

Le schéma d'Euler-Maruyama (annexe du cours, même esprit que le Milstein du TP7) pour simuler une EDS
$\mathrm{d}X_t = \mu(X)\,\mathrm{d}t + \sigma(X)\,\mathrm{d}B_t$.

In [3]:
def EulerMaruyama(x0, mu, sigma, tau, n, rng):
    X = np.zeros(n)
    X[0] = x0
    pas = tau / n
    dB = rng.normal(0, np.sqrt(pas), size=n)
    for i in range(1, n):
        X[i] = X[i-1] + mu(X[i-1]) * pas + sigma(X[i-1]) * dB[i]
    return X

## 3. Une fonction de test (façon TP8)

On simule beaucoup de trajectoires du modèle candidat et on regarde si les statistiques de $X$ (amplitude,
autocorrélation) tombent dans la masse du modèle. Si la $p$-valeur est grande, on ne rejette pas.

In [4]:
def amplitude(x):
    return np.max(np.abs(x))

def autocorr50(x):
    return np.corrcoef(x[:-50], x[50:])[0, 1]

def teste_modele(mu, sigma, x0=0.0, K=300):
    rng = np.random.default_rng(0)
    amp = []; ac = []
    for _ in range(K):
        x = EulerMaruyama(x0, mu, sigma, tau, len(X), rng)
        amp.append(amplitude(x)); ac.append(autocorr50(x))
    amp = np.array(amp); ac = np.array(ac)
    for nom, reel, sim in [('amplitude', amplitude(X), amp), ('autocorr lag50', autocorr50(X), ac)]:
        p = 2 * min(np.mean(sim >= reel), np.mean(sim <= reel))
        print('%-14s  X reel=%.3f   modele[5%%,95%%]=[%.3f, %.3f]   p=%.3f'
              % (nom, reel, np.percentile(sim, 5), np.percentile(sim, 95), p))

## 4. Test du premier candidat naturel, la loi normale (question du TP8)

Le TP8 demande, est-ce un processus gaussien, $X$ suit-il une loi normale ? On teste la marginale.

In [5]:
p_norm = scs.ks_1samp(X, scs.norm(X.mean(), X.std(ddof=1)).cdf).pvalue
print('KS marginale de X contre une normale : p =', round(p_norm, 4))
print('=> rejete. La marginale n est pas normale (X derive lentement, il passe par plusieurs niveaux).')

KS marginale de X contre une normale : p = 0.0
=> rejete. La marginale n est pas normale (X derive lentement, il passe par plusieurs niveaux).


## 5. Candidat 2, une EDS linéaire à retour de moyenne (famille du TP7)

C'est la forme $\mathrm{d}X_t = -\gamma(X_t - m)\,\mathrm{d}t + s\,\mathrm{d}B_t$. On estime les paramètres,
puis on teste.

In [6]:
# estimation rapide (AR1, comme au modele 1)
x0v, x1v = X[:-1], X[1:]
rho = np.cov(x0v, x1v, ddof=1)[0, 1] / np.var(x0v, ddof=1)
m = (np.mean(x1v) - rho * np.mean(x0v)) / (1 - rho)
gamma = -np.log(rho) / dt
s = np.std(x1v - (m + rho * (x0v - m)), ddof=1) / np.sqrt((1 - rho**2) / (2 * gamma))
print('gamma=%.3f  m=%.3f  s=%.3f' % (gamma, m, s))

# le candidat OU : derive lineaire, bruit constant
def mu_ou(x):
    return -gamma * (x - m)
def sigma_ou(x):
    return s

teste_modele(mu_ou, sigma_ou, x0=X[0])

gamma=0.069  m=-0.657  s=0.052
amplitude       X reel=0.707   modele[5%,95%]=[0.447, 0.854]   p=0.760
autocorr lag50  X reel=0.980   modele[5%,95%]=[0.909, 0.985]   p=0.373


Les deux $p$-valeurs sont grandes, donc le modèle linéaire à retour de moyenne n'est pas rejeté. C'est
notre meilleur candidat.

## 6. À toi de tester

Remplis `mon_mu` et `mon_sigma` avec l'EDS que tu veux essayer, puis lance `teste_modele`. Quelques idées
inspirées des TP.
- bruit dépendant de l'état, par exemple `sigma(x) = s * sqrt(1 + x**2)` (façon TP7).
- une dérive non linéaire, par exemple `mu(x) = -gamma * (x - m)**1` que tu peux changer.

In [7]:
def mon_mu(x):
    return -gamma * (x - m)        # change-moi

def mon_sigma(x):
    return s                        # change-moi, par exemple s * np.sqrt(1 + x**2)

teste_modele(mon_mu, mon_sigma, x0=X[0])

amplitude       X reel=0.707   modele[5%,95%]=[0.447, 0.854]   p=0.760
autocorr lag50  X reel=0.980   modele[5%,95%]=[0.909, 0.985]   p=0.373


## 7. Trois candidats à comparer avec l'OU

Chaque candidat teste une hypothèse différente. On regarde si ses $p$-valeurs battent celles de l'OU.

**Candidat A, sans retour de moyenne.** Une marche brownienne avec dérive, $\mathrm{d}X = a\,\mathrm{d}t + s\,\mathrm{d}B$. Pas de rappel. Sert à voir si le retour de moyenne est vraiment nécessaire.

In [ ]:
a = (X[-1] - X[0]) / tau     # derive moyenne constante
def mu_A(x): return a
def sigma_A(x): return s
print('Candidat A (sans retour de moyenne) :')
teste_modele(mu_A, sigma_A, x0=X[0])

**Candidat B, bruit dépendant de l'état (façon TP7).** Même rappel que l'OU, mais $\sigma(x) = s\sqrt{1 + x^2}$, le bruit grossit quand $X$ s'éloigne de zéro.

In [ ]:
def mu_B(x): return -gamma * (x - m)
def sigma_B(x): return s * np.sqrt(1 + x**2)
print('Candidat B (bruit dependant de l etat) :')
teste_modele(mu_B, sigma_B, x0=X[0])

**Candidat C, retour de moyenne non linéaire.** $\mu(x) = -\gamma(x - m) - \beta(x - m)^3$, le rappel devient plus fort quand on s'éloigne.

In [ ]:
beta = 5 * gamma
def mu_C(x): return -gamma * (x - m) - beta * (x - m)**3
def sigma_C(x): return s
print('Candidat C (retour de moyenne non lineaire) :')
teste_modele(mu_C, sigma_C, x0=X[0])

## Conclusion

En traitant $X$ comme dans le TP8 et en testant des EDS façon TP7, le modèle linéaire à retour de moyenne
(l'Ornstein-Uhlenbeck) ressort comme le meilleur. La marginale normale seule est rejetée, mais l'EDS
linéaire reproduit bien les statistiques de $X$.